In [1]:
from pathlib import Path
from datetime import datetime
import polars as pl

In [13]:
class DataLoader:
    def __init__(self, file_path):
        self.file_path = Path(file_path)

    def load(self):
        self._validate_file_path()

        file_extension = self.file_path.suffix.lower()

        if file_extension == ".csv":
            return self._load_csv()

        if file_extension == ".parquet":
            return self._load_parquet()

        if file_extension == ".sqlite":
            return self._load_sqlite()

        raise ValueError(
            f"Неподдерживаемый формат файла: {file_extension}. "
            "Поддерживаются только .csv и .parquet."
        )

    def _validate_file_path(self):
        if not self.file_path.exists():
            raise FileNotFoundError(f"Файл не найден: {self.file_path}")

        if not self.file_path.is_file():
            raise ValueError(f"Указанный путь не является файлом: {self.file_path}")

    def _load_csv(self):
        try:
            separator = self._detect_csv_separator()
            
            return pl.read_csv(
                self.file_path,
                encoding='utf-8',
                separator=separator,
                try_parse_dates=True,
                ignore_errors=True,
                null_values=['', 'NULL', 'null', 'None', 'N/A', 'не указано'],
                # truncate_ragged_lines=True,
                infer_schema_length=10_000,
                # quote_char=None,
                low_memory=True,                
            )
        except Exception as error:
            raise RuntimeError(f"Ошибка при чтении CSV-файла: {error}") from error

    def _load_parquet(self):
        try:
            return pl.read_parquet(self.file_path)
        except Exception as error:
            raise RuntimeError(f"Ошибка при чтении Parquet-файла: {error}") from error

    def _load_sqlite(self):
        import sqlite3
    
        try:
            connection = sqlite3.connect(self.file_path)
    
            tables = pl.read_database(
                query="""
                    SELECT name
                    FROM sqlite_master
                    WHERE type='table'
                    AND name NOT LIKE 'sqlite_%'
                """,
                connection=connection
            )
    
            if tables.height == 0:
                raise ValueError("В базе данных отсутствуют таблицы.")
    
            table_name = tables["name"][0]
    
            df = pl.read_database(
                query=f"SELECT * FROM {table_name}",
                connection=connection
            )
    
            connection.close()
    
            return df
    
        except Exception as error:
            raise RuntimeError(
                f"Ошибка при чтении SQLite-файла: {error}"
            ) from error

    def _detect_csv_separator(self, sample_size=20):
        possible_separators = [
            ",",
            ";",
            "\t",
            "|"
        ]
    
        try:
            with open(self.file_path, "r", encoding="utf-8") as file:
                lines = []
    
                for _ in range(sample_size):
                    line = file.readline()
    
                    if not line:
                        break
    
                    line = line.strip()
    
                    if line:
                        lines.append(line)
    
            if not lines:
                return ","
    
            separator_scores = {}
    
            for separator in possible_separators:
                counts = [line.count(separator) for line in lines]
    
                average_count = sum(counts) / len(counts)
    
                consistency = counts.count(counts[0]) / len(counts)
    
                score = average_count * consistency
    
                separator_scores[separator] = score
    
            best_separator = max(
                separator_scores,
                key=separator_scores.get
            )
    
            return best_separator
    
        except Exception:
            return ","

In [14]:
class ColumnInspector:
    def __init__(self, df):
        self.df = df

    def inspect(self):
        numeric_columns = self._get_numeric_columns()
        boolean_columns = self._get_boolean_columns()
        datetime_columns = self._get_datetime_columns()
        id_like_columns = self._get_id_like_columns()
        text_columns = self._get_text_columns()
    
        datetime_set = set(datetime_columns)
        id_like_set = set(id_like_columns)
        text_set = set(text_columns)
    
        numeric_columns = [
            column for column in numeric_columns
            if column not in datetime_set
            and column not in id_like_set
        ]
    
        boolean_columns = [
            column for column in boolean_columns
            if column not in datetime_set
            and column not in id_like_set
        ]
    
        text_columns = [
            column for column in text_columns
            if column not in datetime_set
            and column not in id_like_set
        ]
    
        categorical_columns = [
            column for column in self._get_categorical_columns()
            if column not in datetime_set
            and column not in id_like_set
            and column not in text_set
        ]
    
        return {
            "numeric": numeric_columns,
            "categorical": categorical_columns,
            "datetime": datetime_columns,
            "boolean": boolean_columns,
            "text": text_columns,
            "id_like": id_like_columns
        }

    def _get_numeric_columns(self):
        numeric_types = {
            pl.Int8,
            pl.Int16,
            pl.Int32,
            pl.Int64,
            pl.UInt8,
            pl.UInt16,
            pl.UInt32,
            pl.UInt64,
            pl.Float32,
            pl.Float64
        }

        return [
            column
            for column, dtype in self.df.schema.items()
            if dtype in numeric_types
        ]

    def _get_categorical_columns(self):
        string_columns = self._get_string_columns()
        text_columns = self._get_text_columns()
        id_columns = self._get_id_like_columns()

        return [
            column
            for column in string_columns
            if column not in text_columns
            and column not in id_columns
        ]

    def _get_datetime_columns(self):
        datetime_columns = []
    
        for column, dtype in self.df.schema.items():
            if dtype == pl.Date or isinstance(dtype, pl.Datetime):
                datetime_columns.append(column)
    
        string_columns = self._get_string_columns()
    
        for column in string_columns:
            sample = self.df[column].drop_nulls().head(50)
    
            if sample.len() == 0:
                continue
    
            parsed_count = 0
    
            try:
                parsed = sample.str.strptime(pl.Datetime, strict=False)
                parsed_count = parsed.drop_nulls().len()
            except Exception:
                try:
                    parsed = sample.str.strptime(pl.Date, strict=False)
                    parsed_count = parsed.drop_nulls().len()
                except Exception:
                    parsed_count = 0
    
            if parsed_count >= sample.len() * 0.7:
                datetime_columns.append(column)
    
        return list(dict.fromkeys(datetime_columns))

    def _get_boolean_columns(self):
        return [
            column
            for column, dtype in self.df.schema.items()
            if dtype == pl.Boolean
        ]

    def _get_text_columns(self):
        text_columns = []

        string_columns = self._get_string_columns()

        for column in string_columns:
            non_null = self.df[column].drop_nulls()

            if non_null.len() == 0:
                continue

            average_length = (
                non_null
                .str.len_chars()
                .mean()
            )

            unique_ratio = (
                non_null.n_unique() / max(non_null.len(), 1)
            )

            if average_length > 50 or unique_ratio > 0.9:
                text_columns.append(column)

        return text_columns

    def _get_id_like_columns(self):
        id_columns = []
    
        strict_id_keywords = [
            "id",
            "uuid",
            "guid"
        ]
    
        technical_keywords = [
            "url",
            "link"
        ]
    
        excluded_code_patterns = [
            "regioncode",
            "federaldistrictcode",
            "codeprofession",
            "codeprofessionalsphere",
            "addresscode",
            "oksocode",
            "oknpocode"
        ]
    
        total_rows = self.df.height
    
        for column in self.df.columns:
            column_lower = column.lower()
    
            normalized_column = (
                column_lower
                .replace("_", "")
                .replace("-", "")
                .replace(" ", "")
            )
    
            if normalized_column in excluded_code_patterns:
                continue
    
            if any(keyword in column_lower for keyword in technical_keywords):
                id_columns.append(column)
                continue
    
            if any(keyword in column_lower for keyword in strict_id_keywords):
                id_columns.append(column)
                continue
    
            non_null = self.df[column].drop_nulls()
    
            if non_null.len() == 0:
                continue
    
            unique_ratio = non_null.n_unique() / max(non_null.len(), 1)
    
            if unique_ratio > 0.995 and non_null.len() > 1000:
                id_columns.append(column)
    
        return list(dict.fromkeys(id_columns))

    def _get_string_columns(self):
        string_types = {
            pl.String,
            pl.Utf8,
            pl.Categorical
        }

        return [
            column
            for column, dtype in self.df.schema.items()
            if dtype in string_types
        ]

In [15]:
class DataAnalyzer:
    def __init__(self, df, columns_info):
        self.df = df
        self.columns_info = columns_info

    def analyze(self):
        return {
            "general_info": self.analyze_general_info(),
            "missing_values": self.analyze_missing_values(),
            "numeric_columns": self.analyze_numeric_columns(),
            "categorical_columns": self.analyze_categorical_columns(),
            "datetime_columns": self.analyze_datetime_columns(),
            "duplicates": self.analyze_duplicates(),
            "correlations": self.analyze_correlations(),
            "quality_issues": self.analyze_quality_issues()
        }

    def analyze_general_info(self):
        rows = self.df.height
        columns = self.df.width

        memory_mb = None

        try:
            memory_mb = round(self.df.estimated_size("mb"), 3)
        except Exception:
            pass

        return {
            "rows": rows,
            "columns": columns,
            "memory_mb": memory_mb,
            "schema": {column: str(dtype) for column, dtype in self.df.schema.items()},
            "column_groups": {
                "numeric": len(self.columns_info.get("numeric", [])),
                "categorical": len(self.columns_info.get("categorical", [])),
                "datetime": len(self.columns_info.get("datetime", [])),
                "boolean": len(self.columns_info.get("boolean", [])),
                "text": len(self.columns_info.get("text", [])),
                "id_like": len(self.columns_info.get("id_like", []))
            }
        }

    def analyze_missing_values(self):
        rows = self.df.height
        result = []

        for column in self.df.columns:
            null_count = self.df[column].null_count()
            null_percent = round(null_count / rows * 100, 2) if rows > 0 else 0

            result.append({
                "column": column,
                "null_count": null_count,
                "null_percent": null_percent
            })

        result = sorted(result, key=lambda item: item["null_percent"], reverse=True)

        critical_columns = [
            item for item in result
            if item["null_percent"] >= 70
        ]

        warning_columns = [
            item for item in result
            if 30 <= item["null_percent"] < 70
        ]

        return {
            "by_columns": result,
            "critical_columns": critical_columns,
            "warning_columns": warning_columns
        }

    def analyze_numeric_columns(self):
        result = []
        skipped_columns = []
    
        for column in self.columns_info.get("numeric", []):
            if self._is_almost_empty_column(column):
                skipped_columns.append(column)
                continue
    
            series = self.df[column].drop_nulls()
    
            if series.len() == 0:
                continue
    
            q1 = series.quantile(0.25)
            q3 = series.quantile(0.75)
            iqr = q3 - q1 if q1 is not None and q3 is not None else None
    
            outliers_count = None
            outliers_percent = None
    
            if iqr is not None:
                lower_bound = q1 - 1.5 * iqr
                upper_bound = q3 + 1.5 * iqr
    
                outliers_count = self.df.filter(
                    (pl.col(column) < lower_bound) | (pl.col(column) > upper_bound)
                ).height
    
                outliers_percent = round(outliers_count / self.df.height * 100, 2) if self.df.height > 0 else 0
    
            result.append({
                "column": column,
                "non_null_count": series.len(),
                "unique_count": series.n_unique(),
                "min": series.min(),
                "max": series.max(),
                "mean": round(series.mean(), 4) if series.mean() is not None else None,
                "median": series.median(),
                "std": round(series.std(), 4) if series.std() is not None else None,
                "zeros_count": self.df.filter(pl.col(column) == 0).height,
                "negative_count": self.df.filter(pl.col(column) < 0).height,
                "outliers_count": outliers_count,
                "outliers_percent": outliers_percent
            })
    
        return {
            "analyzed": result,
            "skipped_almost_empty": skipped_columns
        }

    def analyze_categorical_columns(self):
        result = []
    
        for column in self.columns_info.get("categorical", []):
            cleaned_df = self.df.filter(
                pl.col(column).is_not_null()
                & ~pl.col(column).cast(pl.String).str.strip_chars().str.to_lowercase().is_in(
                    ["", "none", "null", "nan", "[]", "{}", "false"]
                )
            )
    
            series = cleaned_df[column]
    
            if series.len() == 0:
                result.append({
                    "column": column,
                    "non_null_count": 0,
                    "unique_count": 0,
                    "unique_percent": 0,
                    "top_values": [],
                    "empty_like_count": self.df.height
                })
                continue
    
            empty_like_count = self.df.height - cleaned_df.height
    
            value_counts = (
                cleaned_df
                .group_by(column)
                .len()
                .sort("len", descending=True)
                .head(10)
            )
    
            top_values = []
    
            for row in value_counts.iter_rows(named=True):
                value = row[column]
                count = row["len"]
                percent = round(count / self.df.height * 100, 2) if self.df.height > 0 else 0
    
                top_values.append({
                    "value": value,
                    "count": count,
                    "percent": percent
                })
    
            unique_count = series.n_unique()
            unique_percent = round(unique_count / self.df.height * 100, 2) if self.df.height > 0 else 0
    
            result.append({
                "column": column,
                "non_null_count": series.len(),
                "unique_count": unique_count,
                "unique_percent": unique_percent,
                "top_values": top_values,
                "empty_like_count": empty_like_count
            })
    
        return result

    def analyze_datetime_columns(self):
        result = []

        for column in self.columns_info.get("datetime", []):
            series = self.df[column].drop_nulls()

            if series.len() == 0:
                result.append({
                    "column": column,
                    "non_null_count": 0,
                    "min": None,
                    "max": None,
                    "year_distribution": []
                })
                continue

            parsed = self._to_datetime_series(column)

            if parsed is None:
                result.append({
                    "column": column,
                    "non_null_count": series.len(),
                    "min": None,
                    "max": None,
                    "year_distribution": []
                })
                continue

            temp_df = self.df.select(parsed.alias(column)).drop_nulls()

            if temp_df.height == 0:
                result.append({
                    "column": column,
                    "non_null_count": 0,
                    "min": None,
                    "max": None,
                    "year_distribution": []
                })
                continue

            year_distribution_df = (
                temp_df
                .with_columns(pl.col(column).dt.year().alias("year"))
                .group_by("year")
                .len()
                .sort("year")
            )

            result.append({
                "column": column,
                "non_null_count": temp_df.height,
                "min": temp_df[column].min(),
                "max": temp_df[column].max(),
                "year_distribution": year_distribution_df.to_dicts()
            })

        return result

    def analyze_duplicates(self):
        total_rows = self.df.height
        unique_rows = self.df.unique().height
        duplicate_rows = total_rows - unique_rows
        duplicate_percent = round(duplicate_rows / total_rows * 100, 2) if total_rows > 0 else 0

        id_duplicates = []

        for column in self.columns_info.get("id_like", []):
            non_null = self.df[column].drop_nulls()

            if non_null.len() == 0:
                continue

            duplicate_count = non_null.len() - non_null.n_unique()
            duplicate_percent_column = round(duplicate_count / non_null.len() * 100, 2)

            id_duplicates.append({
                "column": column,
                "duplicate_count": duplicate_count,
                "duplicate_percent": duplicate_percent_column
            })

        return {
            "duplicate_rows": duplicate_rows,
            "duplicate_percent": duplicate_percent,
            "id_duplicates": id_duplicates
        }

    def analyze_correlations(self):
        numeric_columns = self.columns_info.get("numeric", [])

        if len(numeric_columns) < 2:
            return {
                "available": False,
                "reason": "Недостаточно числовых колонок для расчета корреляций.",
                "strong_correlations": []
            }

        if len(numeric_columns) > 50:
            return {
                "available": False,
                "reason": "Слишком много числовых колонок для первичного отчета.",
                "strong_correlations": []
            }

        strong_correlations = []

        for i in range(len(numeric_columns)):
            for j in range(i + 1, len(numeric_columns)):
                col_1 = numeric_columns[i]
                col_2 = numeric_columns[j]

                try:
                    corr = self.df.select(pl.corr(col_1, col_2)).item()
                except Exception:
                    corr = None

                if corr is None:
                    continue

                if abs(corr) >= 0.7:
                    strong_correlations.append({
                        "column_1": col_1,
                        "column_2": col_2,
                        "correlation": round(corr, 4)
                    })

        strong_correlations = sorted(
            strong_correlations,
            key=lambda item: abs(item["correlation"]),
            reverse=True
        )

        return {
            "available": True,
            "strong_correlations": strong_correlations[:30]
        }

    def analyze_quality_issues(self):
        issues = {
            "critical": [],
            "warnings": [],
            "info": []
        }

        rows = self.df.height

        for item in self.analyze_missing_values()["critical_columns"]:
            issues["critical"].append(
                f"Колонка '{item['column']}' содержит {item['null_percent']}% пропусков."
            )

        for item in self.analyze_missing_values()["warning_columns"]:
            issues["warnings"].append(
                f"Колонка '{item['column']}' содержит {item['null_percent']}% пропусков."
            )

        for column in self.df.columns:
            non_null = self.df[column].drop_nulls()

            if non_null.len() == 0:
                issues["critical"].append(f"Колонка '{column}' полностью пустая.")
                continue

            unique_count = non_null.n_unique()

            if unique_count == 1:
                issues["warnings"].append(f"Колонка '{column}' является константной.")

            if rows > 0 and unique_count / rows > 0.98:
                issues["info"].append(f"Колонка '{column}' имеет почти уникальные значения и может быть идентификатором.")

        for column in self.columns_info.get("id_like", []):
            issues["info"].append(f"Колонка '{column}' похожа на идентификатор.")

        for column in self.columns_info.get("text", []):
            issues["info"].append(f"Колонка '{column}' похожа на текстовое поле.")

        duplicate_info = self.analyze_duplicates()

        if duplicate_info["duplicate_percent"] > 0:
            issues["warnings"].append(
                f"В датасете найдено {duplicate_info['duplicate_rows']} полных дублей "
                f"({duplicate_info['duplicate_percent']}%)."
            )

        return issues

    def _is_empty_like_value(self, value):
        if value is None:
            return True
    
        value_str = str(value).strip().lower()
    
        empty_like_values = {
            "",
            "none",
            "null",
            "nan",
            "[]",
            "{}",
            "false"
        }
    
        return value_str in empty_like_values

    def _to_datetime_series(self, column):
        dtype = self.df.schema[column]

        if dtype == pl.Date or isinstance(dtype, pl.Date):
            return pl.col(column)

        if dtype == pl.Datetime or isinstance(dtype, pl.Datetime):
            return pl.col(column)

        try:
            return pl.col(column).str.strptime(pl.Datetime, strict=False)
        except Exception:
            try:
                return pl.col(column).str.strptime(pl.Date, strict=False)
            except Exception:
                return None

    def _is_almost_empty_column(self, column, threshold=0.95):
        if self.df.height == 0:
            return True
    
        null_count = self.df[column].null_count()
    
        empty_like_count = 0
    
        if self.df[column].dtype in [pl.String, pl.Utf8, pl.Categorical]:
            empty_like_count = self.df.filter(
                pl.col(column).is_not_null()
                & pl.col(column).cast(pl.String).str.strip_chars().str.to_lowercase().is_in(
                    ["", "none", "null", "nan", "[]", "{}", "false"]
                )
            ).height
    
        empty_total = null_count + empty_like_count
        empty_ratio = empty_total / self.df.height
    
        return empty_ratio >= threshold

In [16]:
class ReportBuilder:
    def __init__(self, analysis_result):
        self.analysis_result = analysis_result
        self.lines = []

    def build(self):
        self.lines = []

        self._add_header()
        self._add_general_info()
        self._add_missing_values()
        self._add_numeric_columns()
        self._add_categorical_columns()
        self._add_datetime_columns()
        self._add_duplicates()
        self._add_correlations()
        self._add_quality_issues()
        self._add_recommendations()

        return "\n".join(self.lines)

    def _add_header(self):
        self.lines.append("=" * 100)
        self.lines.append("ПЕРВИЧНЫЙ ОТЧЕТ О ДАННЫХ")
        self.lines.append("=" * 100)
        self.lines.append(f"Дата формирования: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
        self.lines.append("")

    def _add_section_title(self, title):
        self.lines.append("")
        self.lines.append("-" * 100)
        self.lines.append(title)
        self.lines.append("-" * 100)

    def _add_general_info(self):
        info = self.analysis_result.get("general_info", {})

        self._add_section_title("1. ОБЩАЯ ИНФОРМАЦИЯ")

        self.lines.append(f"Количество строк: {info.get('rows')}")
        self.lines.append(f"Количество колонок: {info.get('columns')}")
        self.lines.append(f"Размер в памяти, MB: {info.get('memory_mb')}")

        self.lines.append("")
        self.lines.append("Группы колонок:")

        column_groups = info.get("column_groups", {})

        for group_name, count in column_groups.items():
            self.lines.append(f"- {group_name}: {count}")

        self.lines.append("")
        self.lines.append("Схема данных:")

        schema = info.get("schema", {})

        for column, dtype in schema.items():
            self.lines.append(f"- {column}: {dtype}")

    def _add_missing_values(self):
        missing = self.analysis_result.get("missing_values", {})

        self._add_section_title("2. ПРОПУСКИ")

        by_columns = missing.get("by_columns", [])

        if not by_columns:
            self.lines.append("Информация о пропусках отсутствует.")
            return

        total_columns_with_missing = sum(1 for item in by_columns if item["null_count"] > 0)

        self.lines.append(f"Колонок с пропусками: {total_columns_with_missing}")

        self.lines.append("")
        self.lines.append("Топ колонок по доле пропусков:")

        for item in by_columns[:20]:
            if item["null_count"] == 0:
                continue

            self.lines.append(
                f"- {item['column']}: {item['null_count']} пропусков "
                f"({item['null_percent']}%)"
            )

        critical_columns = missing.get("critical_columns", [])
        warning_columns = missing.get("warning_columns", [])

        if critical_columns:
            self.lines.append("")
            self.lines.append("Критически разреженные колонки, 70%+ пропусков:")

            for item in critical_columns:
                self.lines.append(f"- {item['column']}: {item['null_percent']}%")

        if warning_columns:
            self.lines.append("")
            self.lines.append("Колонки с высокой долей пропусков, 30–70%:")

            for item in warning_columns:
                self.lines.append(f"- {item['column']}: {item['null_percent']}%")

    def _add_numeric_columns(self):
        numeric_data = self.analysis_result.get("numeric_columns", {})
        numeric = numeric_data.get("analyzed", [])
        skipped = numeric_data.get("skipped_almost_empty", [])
    
        self._add_section_title("3. ЧИСЛОВЫЕ КОЛОНКИ")
    
        if skipped:
            self.lines.append("Почти пустые числовые колонки исключены из подробного анализа:")
    
            for column in skipped:
                self.lines.append(f"- {column}")
    
            self.lines.append("")
    
        if not numeric:
            self.lines.append("Числовые колонки для подробного анализа не найдены.")
            return
    
        for item in numeric:
            self.lines.append("")
            self.lines.append(f"Колонка: {item['column']}")
            self.lines.append(f"- непустых значений: {item['non_null_count']}")
            self.lines.append(f"- уникальных значений: {item['unique_count']}")
            self.lines.append(f"- min: {item['min']}")
            self.lines.append(f"- max: {item['max']}")
            self.lines.append(f"- mean: {item['mean']}")
            self.lines.append(f"- median: {item['median']}")
            self.lines.append(f"- std: {item['std']}")
            self.lines.append(f"- нулевых значений: {item['zeros_count']}")
            self.lines.append(f"- отрицательных значений: {item['negative_count']}")
            self.lines.append(
                f"- выбросов по IQR: {item['outliers_count']} "
                f"({item['outliers_percent']}%)"
            )

    def _add_categorical_columns(self):
        categorical = self.analysis_result.get("categorical_columns", [])

        self._add_section_title("4. КАТЕГОРИАЛЬНЫЕ КОЛОНКИ")

        if not categorical:
            self.lines.append("Категориальные колонки не найдены.")
            return

        for item in categorical:
            self.lines.append("")
            self.lines.append(f"Колонка: {item['column']}")
            self.lines.append(f"- непустых значений: {item['non_null_count']}")
            self.lines.append(f"- уникальных значений: {item['unique_count']}")
            self.lines.append(f"- доля уникальных значений: {item['unique_percent']}%")

            top_values = item.get("top_values", [])

            if top_values:
                self.lines.append("- самые частые значения:")

                for value_info in top_values:
                    self.lines.append(
                        f"  - {value_info['value']}: {value_info['count']} "
                        f"({value_info['percent']}%)"
                    )

    def _add_datetime_columns(self):
        datetime_columns = self.analysis_result.get("datetime_columns", [])

        self._add_section_title("5. ДАТЫ")

        if not datetime_columns:
            self.lines.append("Дата-временные колонки не найдены.")
            return

        for item in datetime_columns:
            self.lines.append("")
            self.lines.append(f"Колонка: {item['column']}")
            self.lines.append(f"- непустых значений: {item['non_null_count']}")
            self.lines.append(f"- минимальная дата: {item['min']}")
            self.lines.append(f"- максимальная дата: {item['max']}")

            year_distribution = item.get("year_distribution", [])

            if year_distribution:
                self.lines.append("- распределение по годам:")

                for row in year_distribution:
                    self.lines.append(f"  - {row['year']}: {row['len']}")

    def _add_duplicates(self):
        duplicates = self.analysis_result.get("duplicates", {})

        self._add_section_title("6. ДУБЛИ")

        self.lines.append(f"Полных дублей строк: {duplicates.get('duplicate_rows')}")
        self.lines.append(f"Доля полных дублей: {duplicates.get('duplicate_percent')}%")

        id_duplicates = duplicates.get("id_duplicates", [])

        if id_duplicates:
            self.lines.append("")
            self.lines.append("Дубли в ID-подобных колонках:")

            for item in id_duplicates:
                self.lines.append(
                    f"- {item['column']}: {item['duplicate_count']} дублей "
                    f"({item['duplicate_percent']}%)"
                )

    def _add_correlations(self):
        correlations = self.analysis_result.get("correlations", {})

        self._add_section_title("7. КОРРЕЛЯЦИИ")

        if not correlations.get("available"):
            self.lines.append(correlations.get("reason", "Корреляции не рассчитаны."))
            return

        strong_correlations = correlations.get("strong_correlations", [])

        if not strong_correlations:
            self.lines.append("Сильные корреляции между числовыми колонками не найдены.")
            return

        self.lines.append("Сильные корреляции, |corr| >= 0.7:")

        for item in strong_correlations:
            self.lines.append(
                f"- {item['column_1']} ↔ {item['column_2']}: {item['correlation']}"
            )

    def _add_quality_issues(self):
        issues = self.analysis_result.get("quality_issues", {})

        self._add_section_title("8. ПРОБЛЕМЫ КАЧЕСТВА ДАННЫХ")

        critical = issues.get("critical", [])
        warnings = issues.get("warnings", [])
        info = issues.get("info", [])

        if not critical and not warnings and not info:
            self.lines.append("Явные проблемы качества данных не обнаружены.")
            return

        if critical:
            self.lines.append("")
            self.lines.append("Критические проблемы:")

            for issue in critical:
                self.lines.append(f"- {issue}")

        if warnings:
            self.lines.append("")
            self.lines.append("Предупреждения:")

            for issue in warnings:
                self.lines.append(f"- {issue}")

        if info:
            self.lines.append("")
            self.lines.append("Информационные замечания:")

            for issue in info[:50]:
                self.lines.append(f"- {issue}")

            if len(info) > 50:
                self.lines.append(f"- ... еще {len(info) - 50} замечаний")

    def _add_recommendations(self):
        self._add_section_title("9. РЕКОМЕНДАЦИИ")

        missing = self.analysis_result.get("missing_values", {})
        duplicates = self.analysis_result.get("duplicates", {})
        correlations = self.analysis_result.get("correlations", {})
        quality_issues = self.analysis_result.get("quality_issues", {})

        recommendations = []

        if missing.get("critical_columns"):
            recommendations.append(
                "Проверить колонки с 70%+ пропусков: чаще всего такие признаки лучше удалить "
                "или использовать только при наличии доменного смысла."
            )

        if missing.get("warning_columns"):
            recommendations.append(
                "Для колонок с 30–70% пропусков нужно выбрать стратегию: удаление, заполнение "
                "или отдельная категория для пропущенных значений."
            )

        if duplicates.get("duplicate_rows", 0) > 0:
            recommendations.append(
                "Проверить полные дубли строк. Если они не несут отдельного смысла, их лучше удалить."
            )

        if correlations.get("available") and correlations.get("strong_correlations"):
            recommendations.append(
                "Проверить сильно коррелирующие числовые признаки. Возможно, часть из них дублирует информацию."
            )

        if quality_issues.get("critical"):
            recommendations.append(
                "Сначала обработать критические проблемы качества данных, затем переходить к построению модели."
            )

        if quality_issues.get("warnings"):
            recommendations.append(
                "Проверить предупреждения: часть из них может указывать на признаки, которые ухудшат качество модели."
            )

        if not recommendations:
            recommendations.append(
                "Серьезных проблем на первичном уровне не найдено. Можно переходить к более детальному EDA."
            )

        for recommendation in recommendations:
            self.lines.append(f"- {recommendation}")

In [17]:
class DataReport:
    def __init__(self, file_path):
        self.file_path = Path(file_path)
        self.df = None
        self.columns_info = None
        self.analysis_result = None
        self.report_text = None

    def generate(self):
        loader = DataLoader(self.file_path)
        self.df = loader.load()

        inspector = ColumnInspector(self.df)
        self.columns_info = inspector.inspect()

        analyzer = DataAnalyzer(self.df, self.columns_info)
        self.analysis_result = analyzer.analyze()

        builder = ReportBuilder(self.analysis_result)
        self.report_text = builder.build()

        return self.report_text

    def save(self, output_path=None):
        if self.report_text is None:
            self.generate()

        if output_path is None:
            output_path = self.file_path.with_name(f"{self.file_path.stem}_report.txt")
        else:
            output_path = Path(output_path)

        output_path.write_text(self.report_text, encoding="utf-8")

        return output_path

    def show(self, max_chars=None):
        if self.report_text is None:
            self.generate()

        if max_chars is None:
            print(self.report_text)
        else:
            print(self.report_text[:max_chars])

In [18]:
DATA_ROOT = "D:\\Projects\\Coding\\Notebooks\\Projects\\labor_market_analysis\\data\\raw\\"
MENDELEY_ROOT = DATA_ROOT + "vacancies_hh_trudvsem_mendeley.csv"
TRUDVSEM_ROOT = DATA_ROOT + "vacancies_trudvsem.csv"
TRUDVSEM_LATEST_ROOT = DATA_ROOT + "vacancies_trudvsem_21_05_26.csv"
HH_KAGGLE_ROOT = DATA_ROOT + "vacancies_hh_kaggle.csv"
HH_GITHUB_ROOT = DATA_ROOT + "vacancies_hh_github.sqlite"

SAVE_ROOT = "D:\\Projects\\Coding\\Notebooks\\Projects\\labor_market_analysis\\data\\info\\"
MENDELEY_SAVE = SAVE_ROOT + "report_hh_trudvsem_mendeley.txt"
TRUDVSEM_SAVE = SAVE_ROOT + "report_trudvsem.txt"
TRUDVSEM_LATEST_SAVE = SAVE_ROOT + "report_trudvsem_latest.txt"
HH_KAGGLE_SAVE = SAVE_ROOT + "report_hh_kaggle.txt"
HH_GITHUB_SAVE = SAVE_ROOT + "report_hh_github.txt"

In [12]:
report = DataReport(TRUDVSEM_LATEST_ROOT)
text = report.generate()
report.save(TRUDVSEM_LATEST_SAVE)

WindowsPath('D:/Projects/Coding/Notebooks/Projects/labor_market_analysis/data/info/report_trudvsem_latest.txt')

In [8]:
report = DataReport(TRUDVSEM_ROOT)
text = report.generate()
report.save(TRUDVSEM_SAVE)

WindowsPath('D:/Projects/Coding/ML/SamsungML/data/JobData/info/report_trudvsem.txt')

In [19]:
report = DataReport(MENDELEY_ROOT)
text = report.generate()
report.save(MENDELEY_SAVE)

WindowsPath('D:/Projects/Coding/Notebooks/Projects/labor_market_analysis/data/info/report_hh_trudvsem_mendeley.txt')

In [20]:
report = DataReport(HH_KAGGLE_ROOT)
text = report.generate()
report.save(HH_KAGGLE_SAVE)

WindowsPath('D:/Projects/Coding/Notebooks/Projects/labor_market_analysis/data/info/report_hh_kaggle.txt')

In [21]:
report = DataReport(HH_GITHUB_ROOT)
text = report.generate()
report.save(HH_GITHUB_SAVE)

WindowsPath('D:/Projects/Coding/Notebooks/Projects/labor_market_analysis/data/info/report_hh_github.txt')